In [ ]:
LETTER_MAP = {"A":5,"B":4,"C":3,"D":2,"E":1,"F":1}
DESCRIPTIVE_MAP = {"excellent":5,"very good":4,"good":3,"fair":2,"poor":1,"great":5,"average":3,"bad":1,"terrible":1,"perfect":5,"outstanding":5,"exceptional":5,"satisfactory":3,"inadequate":1,"mediocre":2}

def parse_score(text, variant):
    if not text: return None
    t = text.strip().lower()
    nums = re.findall(r"\b([1-5])\b", t)
    let = re.search(r"\b([a-f])\b", t)
    desc = None
    for w,s in sorted(DESCRIPTIVE_MAP.items(), key=lambda x:-len(x[0])):
        if w in t: desc = float(s); break
    if variant == "numeric":
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return desc
    elif variant == "letter":
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        if nums: return float(nums[0])
        return desc
    elif variant == "descriptive":
        if desc: return desc
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return None
    else:
        if nums: return float(nums[0])
        if let: return float(LETTER_MAP.get(let.group(1).upper(),0))
        return desc

def score_model(m, tok, p, var, dev):
    inp = tok(p, return_tensors="pt", truncation=True, max_length=1024).to(dev)
    with torch.no_grad():
        o = m.generate(**inp, max_new_tokens=20, temperature=0.0, do_sample=False, pad_token_id=tok.eos_token_id)
    f = tok.decode(o[0], skip_special_tokens=True)
    g = f[len(p):].strip()
    if "### Score:" in g:
        s = parse_score(g.split("### Score:")[-1].split("###")[0].strip(), var)
        if s and 1<=s<=5: return s
    s = parse_score(g, var)
    if s and 1<=s<=5: return s
    x = re.findall(r"### Score:?\s*([1-5])", f)
    if x: return float(x[-1])
    return None
print("Parser ready")
